# Silver Products - SCD Type 2 Price History

Builds a Slowly Changing Dimension Type 2 (SCD2) product table in `product.silver.products` from the bronze orders data. Each product's price change creates a new row with `start_date` and `end_date` to track the full price history over time.

In [0]:
%sql
-- Create the SCD2 products table in silver schema
CREATE TABLE IF NOT EXISTS product.silver.products (
    product_id    INT       NOT NULL,
    product_name  STRING    NOT NULL,
    price         INT       NOT NULL,
    start_date    DATE      NOT NULL,
    end_date      DATE
)
USING DELTA;

-- Build SCD2 from bronze orders:
-- 1. Find the effective start date for each (product, price) combination
--    (first day of the month the price first appeared)
-- 2. Use LEAD to determine end_date = day before the next price period starts
-- 3. Latest price gets NULL end_date (active record)
WITH price_periods AS (
    SELECT
        product_id,
        product_name,
        product_price AS price,
        DATE_TRUNC('month', MIN(order_date))::DATE AS start_date
    FROM product.bronze.orders
    GROUP BY product_id, product_name, product_price
),
scd2_with_next AS (
    SELECT
        product_id,
        product_name,
        price,
        start_date,
        LEAD(start_date) OVER (
            PARTITION BY product_id
            ORDER BY start_date
        ) AS next_start_date
    FROM price_periods
)
INSERT INTO product.silver.products
SELECT
    product_id,
    product_name,
    price,
    start_date,
    CASE
        WHEN next_start_date IS NOT NULL
            THEN DATE_SUB(next_start_date, 1)
        ELSE NULL
    END AS end_date
FROM scd2_with_next
ORDER BY product_id, start_date;

num_affected_rows,num_inserted_rows
15,15


In [0]:
%sql
-- Verify the SCD2 product table
SELECT * FROM product.silver.products
ORDER BY product_id, start_date;

product_id,product_name,price,start_date,end_date
1,tea,10,2026-04-01,2026-04-30
1,tea,20,2026-05-01,2026-05-31
1,tea,30,2026-06-01,null
2,coffee,10,2026-04-01,2026-04-30
2,coffee,20,2026-05-01,2026-05-31
2,coffee,30,2026-06-01,null
3,soap,10,2026-04-01,2026-04-30
3,soap,20,2026-05-01,2026-05-31
3,soap,30,2026-06-01,null
4,paste,10,2026-04-01,2026-04-30
